# PAP Emulation - JAX GPU/TPU Ensemble

Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary. Licensed under GNU GPL v3.0. Third-party dependencies retain their own licenses.

A vectorized ODE parameter ensemble with an analytical reference. Demonstrates JAX/XLA GPU/TPU compute and records actual device use. It does not imply every domain solver is GPU accelerated.

**Author:** Abhishek Choudhary · **Affiliation:** AyeAI · **ORCID:** [0009-0002-0684-8320](https://orcid.org/0009-0002-0684-8320)

**Deployment:** browser-only platform; this notebook runs in a user-controlled Colab runtime. GPU/TPU availability is variable and not guaranteed. **Evidence boundary:** results remain `SIMULATED` and `UNVERIFIED`; this is not HIL or physical verification.

## 0. Runtime, security and provenance
Run cells in order. The notebook records backend versions, solver settings, runtime device, model configuration, timestamps and SHA-256 job/result digests. Results must be imported through the separate PAP Colab Bridge. Never upload untrusted native FMUs or binaries.

In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.
# AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
# Embedded in each notebook so notebooks remain independently runnable.
PAP_BRIDGE_VERSION = '1.0.0'
PAP_JOB_SCHEMA = 'pap.emulation.job/1.0'
PAP_RESULT_SCHEMA = 'pap.emulation.result/1.0'

def pap_canonical(value):
    import json
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False)

def pap_digest(text):
    import hashlib
    return 'sha256:' + hashlib.sha256(text.encode('utf-8')).hexdigest()

def pap_make_job(backend_id, operation, inputs):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_JOB_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'createdAt': datetime.now(timezone.utc).isoformat(),
        'target': {'backendId': backend_id, 'backendVersion': None, 'operation': operation, 'requestedDevice': 'auto'},
        'sir': {'header': {'id': 'colab-reference-fixture', 'irVersion': '1.0.0'}, 'components': []},
        'inputs': inputs,
        'observations': None,
        'configuration': {},
        'provenance': {'source': 'COLAB_REFERENCE_FIXTURE', 'sourceVersion': '1.0.0', 'license': 'GPL-3.0-only'}
    }
    canonical = pap_canonical(body)
    digest = pap_digest(canonical)
    return {**body, 'jobId': 'papjob-' + digest[7:27], 'digest': digest, 'canonicalPayload': canonical}

def pap_verify_job(job, expected_backend):
    import json
    if not isinstance(job, dict) or job.get('schema') != PAP_JOB_SCHEMA:
        raise ValueError('Unsupported/missing PAP job schema')
    payload = job.get('canonicalPayload')
    if not isinstance(payload, str): raise ValueError('Job canonicalPayload missing')
    parsed = json.loads(payload)
    body = {k:v for k,v in job.items() if k not in ('jobId','digest','canonicalPayload')}
    if pap_canonical(parsed) != pap_canonical(body): raise ValueError('Job payload/body mismatch')
    digest = pap_digest(payload)
    if digest != job.get('digest') or job.get('jobId') != 'papjob-' + digest[7:27]:
        raise ValueError('Job digest/jobId verification failed')
    if job.get('target',{}).get('backendId') != expected_backend:
        raise ValueError('Job targets a different backend')
    if not isinstance(job.get('inputs'), dict): raise ValueError('Job inputs must be an object')
    return job

def pap_get_job(backend_id, operation, default_inputs, use_demo=True):
    if use_demo:
        return pap_make_job(backend_id, operation, default_inputs), True
    from google.colab import files
    import json
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith('.json')]
    if len(candidates) != 1: raise ValueError('Upload exactly one browser-exported PAP job .json')
    return pap_verify_job(json.loads(uploaded[candidates[0]].decode('utf-8')), backend_id), False

def pap_make_result(job, backend_id, backend_version, outputs, diagnostics, device='cpu'):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_RESULT_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'jobId': job['jobId'],
        'jobDigest': job['digest'],
        'backend': {'id': backend_id, 'version': str(backend_version)},
        'execution': {'status': 'EXECUTED', 'runtime': 'COLAB', 'startedAt': None,
                      'finishedAt': datetime.now(timezone.utc).isoformat(), 'device': str(device)},
        'outputs': outputs,
        'diagnostics': diagnostics,
        'fidelity': 'SIMULATED',
        'physicalVerification': False,
        'evidenceState': 'UNVERIFIED',
        'claims': {'verified': False, 'hil': False, 'physical': False}
    }
    canonical = pap_canonical(body)
    return {**body, 'canonicalPayload': canonical, 'resultDigest': pap_digest(canonical)}

def pap_export(job, result, is_demo, result_filename='pap-emulation-result.json'):
    import json
    with open(result_filename, 'w', encoding='utf-8') as stream:
        json.dump(result, stream, indent=2, ensure_ascii=False, allow_nan=False)
    if is_demo:
        with open('pap-emulation-job.json', 'w', encoding='utf-8') as stream:
            json.dump(job, stream, indent=2, ensure_ascii=False, allow_nan=False)
    try:
        from google.colab import files
        files.download(result_filename)
        if is_demo: files.download('pap-emulation-job.json')
    except ImportError:
        print('Saved locally:', result_filename, '(download manually outside Colab)')
    print('Result remains SIMULATED / UNVERIFIED. Import it through the PAP Colab Bridge.')


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
# Do not reinstall JAX blindly: preserve Colab's accelerator-enabled JAX build.

In [ ]:
import jax, jax.numpy as jnp, time
print({'jax':jax.__version__,'devices':[str(d) for d in jax.devices()],'backend':jax.default_backend()})
PAP_USE_DEMO_JOB=True
default_inputs={'ensembleSize':100000,'steps':200,'dt':0.005,'kMin':0.1,'kMax':5.0}
job,job_is_demo=pap_get_job('jax.xla','jax.ode_ensemble',default_inputs,PAP_USE_DEMO_JOB)

## 1. Vectorized RK4 ensemble
The system is `dx/dt=-k*x+1` with x(0)=0. The closed-form solution is `(1-exp(-k*t))/k`. The notebook compares the vectorized fixed-step RK4 result to that reference over a parameter ensemble.

In [ ]:
inputs=job['inputs']; N=int(inputs.get('ensembleSize',100000)); steps=int(inputs.get('steps',200)); dt=float(inputs.get('dt',0.005)); kmin=float(inputs.get('kMin',0.1)); kmax=float(inputs.get('kMax',5.0))
if N<1 or N>2_000_000 or steps<1 or steps>1_000_000 or dt<=0 or not 0<kmin<kmax: raise ValueError('Invalid ensemble parameters')
k=jnp.linspace(kmin,kmax,N,dtype=jnp.float32)
def rhs(x,k): return -k*x+1.0
def rk4(x,k,dt):
 a=rhs(x,k); b=rhs(x+0.5*dt*a,k); c=rhs(x+0.5*dt*b,k); d=rhs(x+dt*c,k)
 return x+(dt/6)*(a+2*b+2*c+d)
def integrate(k): return jax.lax.fori_loop(0,steps,lambda i,x:rk4(x,k,dt),jnp.zeros_like(k))
compiled=jax.jit(integrate); started=time.time(); xfinal=compiled(k); xfinal.block_until_ready(); elapsed=time.time()-started
if not bool(jnp.isfinite(xfinal).all()): raise RuntimeError('Non-finite state')
reference=(1-jnp.exp(-k*steps*dt))/k; max_error=float(jnp.max(jnp.abs(xfinal-reference)))
print({'ensembleSize':N,'steps':steps,'elapsedSeconds':elapsed,'maxAbsError':max_error,'backend':jax.default_backend()}); assert max_error<3e-5

In [ ]:
outputs=[{'sequence':0,'time':float(steps*dt),'values':{'ensembleMin':float(jnp.min(xfinal)),'ensembleMax':float(jnp.max(xfinal)),'ensembleMean':float(jnp.mean(xfinal))}}]
result=pap_make_result(job,'jax.xla',jax.__version__,outputs,{'ensembleSize':N,'steps':steps,'dt':dt,'maxAbsErrorAgainstAnalytical':max_error,'elapsedSeconds':elapsed,'backend':jax.default_backend(),'devices':[str(d) for d in jax.devices()]},jax.default_backend())
pap_export(job,result,job_is_demo,'pap-jax-result.json')